# Phishing Email Kill Chain Graph

> **Platform:** Microsoft Sentinel Custom Graph | **Domain:** Email Security
> **Data Sources:** EmailEvents, EmailUrlInfo, UrlClickEvents, EmailAttachmentInfo, ThreatIntelIndicators, DeviceFileEvents, DeviceProcessEvents
> **Nodes:** 10 (Email, Sender, User, Url, Domain, Attachment, Process, Device, ThreatIndicator_Domain, ThreatIndicator_Url)
> **Edges:** 12 (Sent, ReceivedEmail, ContainsUrl, HasDomain, HasAttachment, ClickedUrl, StoredOn, DownloadedFile, TriggeredProcess, OnDevice, DomainRelatedTo, UrlRelatedTo)

Connects phishing emails through sender tracking, URL clicks, domain grouping, and threat intelligence correlation to endpoint process execution and file downloads. Enables full kill chain visibility from email delivery through endpoint compromise with TI-enriched domain/URL analysis.

## Environment & Configuration

Import required packages and configure the Sentinel Data Lake connection.

In [ ]:
# OPTIONAL: Adjust the lookback window (default: 7 days). Use 1 for IR, 14 for hunting, 30+ for historical.
TIME_WINDOW_DAYS = 1

# REQUIRED: Set your Microsoft Sentinel workspace name before running this notebook
workspace_name = "<YOUR_WORKSPACE_NAME>"  # Replace with your Sentinel workspace name

import logging

# Set the logging level for the entire package
logging.getLogger('sentinel_graph').setLevel(logging.INFO)
print(f"Logging level set to: {logging.getLevelName(logging.getLogger('sentinel_graph').level)} and above")


In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph import GraphSpecBuilder, Graph
from pyspark.sql.functions import (
    col, lit, expr, count, first, lower, trim, when, concat, concat_ws,
    countDistinct
)

lake_provider = MicrosoftSentinelProvider(spark)

# Helper function to extract PySpark DataFrame from CustomDataFrame wrapper
def to_spark_df(df):
    return df.df if hasattr(df, 'df') else df

print(f"Reading data from workspace: {workspace_name}")
print(f"Lookback period: {TIME_WINDOW_DAYS} days")

# =============================================================================
# Read EmailEvents
# =============================================================================
email_events_df = (
    to_spark_df(lake_provider.read_table("EmailEvents", workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('NetworkMessageId').isNotNull())
    .select(
        "TimeGenerated", "NetworkMessageId", "Subject", "SenderFromAddress",
        "SenderDisplayName", "RecipientEmailAddress", "DeliveryAction",
        "DeliveryLocation", "ThreatTypes", "ThreatNames", "DetectionMethods",
        "InternetMessageId"
    )
).persist()

# =============================================================================
# Read EmailUrlInfo
# =============================================================================
email_url_info_df = (
    to_spark_df(lake_provider.read_table('EmailUrlInfo', workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('NetworkMessageId').isNotNull())
    .select("TimeGenerated", "NetworkMessageId", "Url", "UrlDomain", "UrlLocation")
).persist()

# =============================================================================
# Read UrlClickEvents
# =============================================================================
url_click_events_df = (
    to_spark_df(lake_provider.read_table('UrlClickEvents', workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .select(
        "TimeGenerated", "NetworkMessageId", "Url", "AccountUpn",
        "IsClickedThrough", "ActionType", "Workload", "IPAddress", "ThreatTypes"
    )
).persist()

# =============================================================================
# Read EmailAttachmentInfo
# =============================================================================
email_attachment_df = (
    to_spark_df(lake_provider.read_table('EmailAttachmentInfo', workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('NetworkMessageId').isNotNull())
    .select(
        "TimeGenerated", "NetworkMessageId", "SenderFromAddress",
        "RecipientEmailAddress", "FileName", "FileType", "SHA256",
        "FileSize", "ThreatTypes", "ThreatNames"
    )
).persist()

# =============================================================================
# Read DeviceFileEvents
# =============================================================================
device_file_df = (
    to_spark_df(lake_provider.read_table('DeviceFileEvents', workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('ActionType').isin('FileCreated', 'FileModified'))
    .select(
        "TimeGenerated", "DeviceName", "DeviceId", "ActionType", "FileName",
        "FolderPath", "SHA256", "InitiatingProcessFileName",
        "InitiatingProcessAccountUpn", "FileOriginUrl"
    )
).persist()

# =============================================================================
# Read DeviceProcessEvents
# =============================================================================
device_process_df = (
    to_spark_df(lake_provider.read_table('DeviceProcessEvents', workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
    .filter(col('ActionType') == 'ProcessCreated')
    .select(
        "TimeGenerated", "DeviceName", "DeviceId", "ActionType", "FileName",
        "FolderPath", "ProcessCommandLine", "InitiatingProcessFileName",
        "InitiatingProcessCommandLine", "AccountUpn", "SHA256"
    )
).persist()

# =============================================================================
# Read ThreatIntelIndicators
# =============================================================================
ti_df = (
    to_spark_df(lake_provider.read_table('ThreatIntelIndicators', workspace_name))
    .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS * 2} DAYS"))
    .select("ObservableKey", "ObservableValue", "Confidence", "Tags")
).persist()

# Split into domain and URL indicators
ti_domains_df = (
    ti_df
    .filter(col("ObservableKey") == "domain-name:value")
    .select(col("ObservableValue").alias("TIDomain"), "Confidence", "Tags")
    .dropDuplicates(["TIDomain"])
)

ti_urls_df = (
    ti_df
    .filter(col("ObservableKey") == "url:value")
    .select(col("ObservableValue").alias("TIUrl"), "Confidence", "Tags")
    .dropDuplicates(["TIUrl"])
)

# Validation counts
print(f"EmailEvents: {email_events_df.count()}")
print(f"EmailUrlInfo: {email_url_info_df.count()}")
print(f"UrlClickEvents: {url_click_events_df.count()}")
print(f"EmailAttachmentInfo: {email_attachment_df.count()}")
print(f"DeviceFileEvents: {device_file_df.count()}")
print(f"DeviceProcessEvents: {device_process_df.count()}")
print(f"ThreatIntelIndicators: {ti_df.count()} (Domains: {ti_domains_df.count()}, URLs: {ti_urls_df.count()})")

In [ ]:
# =============================================================================
# NODE DATAFRAMES (10 types)
# =============================================================================

# --- Email nodes ---
email_node_df = (
    email_events_df
    .select(
        col("NetworkMessageId").alias("EmailId"),
        "Subject", "SenderFromAddress", "SenderDisplayName",
        "DeliveryAction", "DeliveryLocation", "ThreatTypes", "TimeGenerated"
    )
    .dropDuplicates(["EmailId"])
    .withColumn("nodeType", lit("Email"))
)

# --- Sender nodes ---
sender_node_df = (
    email_events_df
    .select(
        col("SenderFromAddress").alias("SenderEmail"),
        "SenderDisplayName"
    )
    .dropDuplicates(["SenderEmail"])
    .withColumn("nodeType", lit("Sender"))
)

# --- User nodes (recipients + clickers) ---
user_from_emails = email_events_df.select(
    lower(trim(col("RecipientEmailAddress"))).alias("UserEmail")
).distinct()

user_from_clicks = url_click_events_df.select(
    lower(trim(col("AccountUpn"))).alias("UserEmail")
).distinct()

user_node_df = (
    user_from_emails.union(user_from_clicks)
    .distinct()
    .filter(col("UserEmail").isNotNull())
    .withColumn("nodeType", lit("User"))
)

# --- URL nodes ---
url_node_df = (
    email_url_info_df
    .select(col("Url").alias("UrlId"), "UrlDomain", "UrlLocation")
    .dropDuplicates(["UrlId"])
    .withColumn("nodeType", lit("Url"))
)

# --- Domain nodes ---
domain_node_df = (
    email_url_info_df
    .select(col("UrlDomain").alias("DomainId"))
    .filter(col("DomainId").isNotNull())
    .dropDuplicates(["DomainId"])
    .withColumn("nodeType", lit("Domain"))
)

# --- Attachment nodes (with rarity scoring) ---
attachment_node_df = (
    email_attachment_df
    .filter(col("SHA256").isNotNull())
    .select(
        col("SHA256").alias("AttachmentId"),
        "FileName", "FileType", "FileSize", "ThreatTypes", "ThreatNames"
    )
    .dropDuplicates(["AttachmentId"])
    .withColumn("nodeType", lit("Attachment"))
)

# --- Process nodes ---
process_node_df = (
    device_process_df
    .withColumn("ProcessId",
        concat(col("DeviceName"), lit("_"), col("FileName"), lit("_"), col("TimeGenerated")))
    .select(
        "ProcessId", "FileName", "ProcessCommandLine",
        "InitiatingProcessFileName", "InitiatingProcessCommandLine",
        "DeviceName", "TimeGenerated"
    )
    .dropDuplicates(["ProcessId"])
    .withColumn("nodeType", lit("Process"))
)

# --- Device nodes ---
device_node_df = (
    device_process_df
    .select(col("DeviceName").alias("DeviceId"), "DeviceName")
    .dropDuplicates(["DeviceId"])
    .withColumn("nodeType", lit("Device"))
)

# --- ThreatIndicator_Domain nodes ---
ti_domain_node_df = (
    ti_domains_df
    .select("TIDomain", "Confidence", "Tags")
    .withColumn("nodeType", lit("ThreatIndicator_Domain"))
)

# --- ThreatIndicator_Url nodes ---
ti_url_node_df = (
    ti_urls_df
    .select("TIUrl", "Confidence", "Tags")
    .withColumn("nodeType", lit("ThreatIndicator_Url"))
)

# =============================================================================
# EDGE DATAFRAMES (12 types)
# =============================================================================

# --- Sent: Sender -> Email ---
sent_edge_df = (
    email_events_df
    .select(
        col("SenderFromAddress").alias("SenderEmail"),
        col("NetworkMessageId").alias("EmailId"),
        "TimeGenerated"
    )
    .withColumn("edgeType", lit("Sent"))
    .withColumn("EdgeKey", concat_ws("_", col("SenderEmail"), col("EmailId")))
)

# --- ReceivedEmail: Email -> User ---
received_email_edge_df = (
    email_events_df
    .select(
        col("NetworkMessageId").alias("EmailId"),
        lower(trim(col("RecipientEmailAddress"))).alias("UserEmail"),
        "DeliveryLocation", "DeliveryAction", "TimeGenerated"
    )
    .withColumn("edgeType", lit("ReceivedEmail"))
    .withColumn("EdgeKey", concat_ws("_", col("EmailId"), col("UserEmail")))
)

# --- ContainsUrl: Email -> URL ---
contains_url_edge_df = (
    email_url_info_df
    .select(
        col("NetworkMessageId").alias("EmailId"),
        col("Url").alias("UrlId"),
        "UrlLocation", "TimeGenerated"
    )
    .withColumn("edgeType", lit("ContainsUrl"))
    .withColumn("EdgeKey", concat_ws("_", col("EmailId"), col("UrlId")))
)

# --- HasDomain: Email -> Domain ---
has_domain_edge_df = (
    email_url_info_df
    .select(
        col("NetworkMessageId").alias("EmailId"),
        col("UrlDomain").alias("DomainId"),
        "TimeGenerated"
    )
    .filter(col("DomainId").isNotNull())
    .withColumn("edgeType", lit("HasDomain"))
    .withColumn("EdgeKey", concat_ws("_", col("EmailId"), col("DomainId")))
)

# --- HasAttachment: Email -> Attachment ---
has_attachment_edge_df = (
    email_attachment_df
    .filter(col("SHA256").isNotNull())
    .select(
        col("NetworkMessageId").alias("EmailId"),
        col("SHA256").alias("AttachmentId"),
        "FileName", "TimeGenerated"
    )
    .withColumn("edgeType", lit("HasAttachment"))
    .withColumn("EdgeKey", concat_ws("_", col("EmailId"), col("AttachmentId")))
)

# --- ClickedUrl: User -> URL ---
clicked_url_edge_df = (
    url_click_events_df
    .select(
        lower(trim(col("AccountUpn"))).alias("UserEmail"),
        col("Url").alias("UrlId"),
        "IsClickedThrough", "ActionType", "IPAddress", "TimeGenerated"
    )
    .withColumn("edgeType", lit("ClickedUrl"))
    .withColumn("EdgeKey", concat_ws("_", col("UserEmail"), col("UrlId"), col("TimeGenerated").cast("string")))
)

# --- StoredOn: Attachment -> Device (Outlook file extraction) ---
stored_on_edge_df = (
    device_file_df
    .filter(col("SHA256").isNotNull())
    .filter(col("InitiatingProcessFileName") == "outlook.exe")
    .filter(col("ActionType") == "FileCreated")
    .select(
        col("SHA256").alias("AttachmentId"),
        col("DeviceName").alias("DeviceId"),
        "TimeGenerated"
    )
    .dropDuplicates(["AttachmentId", "DeviceId"])
    .withColumn("edgeType", lit("StoredOn"))
    .withColumn("EdgeKey", concat_ws("_", col("AttachmentId"), col("DeviceId")))
)

# --- DownloadedFile: User -> Attachment ---
downloaded_file_edge_df = (
    device_file_df
    .filter(col("SHA256").isNotNull())
    .select(
        lower(trim(col("InitiatingProcessAccountUpn"))).alias("UserEmail"),
        col("SHA256").alias("AttachmentId"),
        "FileName", "DeviceName", "FileOriginUrl", "TimeGenerated"
    )
    .withColumn("edgeType", lit("DownloadedFile"))
    .withColumn("EdgeKey", concat_ws("_", col("UserEmail"), col("AttachmentId"), col("TimeGenerated").cast("string")))
)

# --- TriggeredProcess: Attachment -> Process ---
triggered_process_edge_df = (
    device_process_df
    .filter(col("SHA256").isNotNull())
    .withColumn("ProcessId",
        concat(col("DeviceName"), lit("_"), col("FileName"), lit("_"), col("TimeGenerated")))
    .select(
        col("SHA256").alias("AttachmentId"),
        "ProcessId",
        "FileName", "ProcessCommandLine", "TimeGenerated"
    )
    .withColumn("edgeType", lit("TriggeredProcess"))
    .withColumn("EdgeKey", concat_ws("_", col("AttachmentId"), col("ProcessId")))
)

# --- OnDevice: Process -> Device ---
on_device_edge_df = (
    device_process_df
    .withColumn("ProcessId",
        concat(col("DeviceName"), lit("_"), col("FileName"), lit("_"), col("TimeGenerated")))
    .select(
        "ProcessId",
        col("DeviceName").alias("DeviceId"),
        "TimeGenerated"
    )
    .withColumn("edgeType", lit("OnDevice"))
    .withColumn("EdgeKey", concat_ws("_", col("ProcessId"), col("DeviceId")))
)

# --- DomainRelatedTo: ThreatIndicator_Domain -> Domain ---
domain_related_edge_df = (
    ti_domains_df
    .join(
        domain_node_df.select("DomainId"),
        ti_domains_df["TIDomain"] == domain_node_df["DomainId"],
        "inner"
    )
    .select(col("TIDomain"), col("DomainId"), "Confidence")
    .withColumn("edgeType", lit("DomainRelatedTo"))
    .withColumn("EdgeKey", concat_ws("_", col("TIDomain"), col("DomainId")))
)

# --- UrlRelatedTo: ThreatIndicator_Url -> URL ---
url_related_edge_df = (
    ti_urls_df
    .join(
        url_node_df.select("UrlId"),
        ti_urls_df["TIUrl"] == url_node_df["UrlId"],
        "inner"
    )
    .select(col("TIUrl"), col("UrlId"), "Confidence")
    .withColumn("edgeType", lit("UrlRelatedTo"))
    .withColumn("EdgeKey", concat_ws("_", col("TIUrl"), col("UrlId")))
)

# =============================================================================
# Validation counts
# =============================================================================
print(f"Nodes - Email: {email_node_df.count()}, Sender: {sender_node_df.count()}, User: {user_node_df.count()}")
print(f"Nodes - Url: {url_node_df.count()}, Domain: {domain_node_df.count()}, Attachment: {attachment_node_df.count()}")
print(f"Nodes - Process: {process_node_df.count()}, Device: {device_node_df.count()}")
print(f"Nodes - TI_Domain: {ti_domain_node_df.count()}, TI_Url: {ti_url_node_df.count()}")
print(f"Edges - Sent: {sent_edge_df.count()}, ReceivedEmail: {received_email_edge_df.count()}")
print(f"Edges - ContainsUrl: {contains_url_edge_df.count()}, HasDomain: {has_domain_edge_df.count()}")
print(f"Edges - HasAttachment: {has_attachment_edge_df.count()}, ClickedUrl: {clicked_url_edge_df.count()}")
print(f"Edges - StoredOn: {stored_on_edge_df.count()}, DownloadedFile: {downloaded_file_edge_df.count()}")
print(f"Edges - TriggeredProcess: {triggered_process_edge_df.count()}, OnDevice: {on_device_edge_df.count()}")
print(f"Edges - DomainRelatedTo: {domain_related_edge_df.count()}, UrlRelatedTo: {url_related_edge_df.count()}")

## Graph Assembly

Building the graph:
- **10 Node Types**: Email, Sender, User, Url, Domain, Attachment, Process, Device, ThreatIndicator_Domain, ThreatIndicator_Url
- **12 Edge Types**: Sent, ReceivedEmail, ContainsUrl, HasDomain, HasAttachment, ClickedUrl, StoredOn, DownloadedFile, TriggeredProcess, OnDevice, DomainRelatedTo, UrlRelatedTo

In [ ]:
# =============================================================================
# Build Graph Specification
# =============================================================================

phishing_killchain_graph = (GraphSpecBuilder.start()

    # ===================== NODES (10 types) =====================

    .add_node("Email")
    .from_dataframe(email_node_df)
    .with_columns(
        "EmailId", "Subject", "SenderFromAddress", "SenderDisplayName",
        "DeliveryAction", "DeliveryLocation", "ThreatTypes", "TimeGenerated", "nodeType",
        key="EmailId", display="Subject"
    )

    .add_node("Sender")
    .from_dataframe(sender_node_df)
    .with_columns(
        "SenderEmail", "SenderDisplayName", "nodeType",
        key="SenderEmail", display="SenderEmail"
    )

    .add_node("User")
    .from_dataframe(user_node_df)
    .with_columns(
        "UserEmail", "nodeType",
        key="UserEmail", display="UserEmail"
    )

    .add_node("Url")
    .from_dataframe(url_node_df)
    .with_columns(
        "UrlId", "UrlDomain", "UrlLocation", "nodeType",
        key="UrlId", display="UrlDomain"
    )

    .add_node("Domain")
    .from_dataframe(domain_node_df)
    .with_columns(
        "DomainId", "nodeType",
        key="DomainId", display="DomainId"
    )

    .add_node("Attachment")
    .from_dataframe(attachment_node_df)
    .with_columns(
        "AttachmentId", "FileName", "FileType", "FileSize", "ThreatTypes", "ThreatNames",
        "nodeType",
        key="AttachmentId", display="FileName"
    )

    .add_node("Process")
    .from_dataframe(process_node_df)
    .with_columns(
        "ProcessId", "FileName", "ProcessCommandLine", "InitiatingProcessFileName",
        "InitiatingProcessCommandLine", "DeviceName", "TimeGenerated", "nodeType",
        key="ProcessId", display="FileName"
    )

    .add_node("Device")
    .from_dataframe(device_node_df)
    .with_columns(
        "DeviceId", "DeviceName", "nodeType",
        key="DeviceId", display="DeviceName"
    )

    .add_node("ThreatIndicator_Domain")
    .from_dataframe(ti_domain_node_df)
    .with_columns(
        "TIDomain", "Confidence", "Tags", "nodeType",
        key="TIDomain", display="TIDomain"
    )

    .add_node("ThreatIndicator_Url")
    .from_dataframe(ti_url_node_df)
    .with_columns(
        "TIUrl", "Confidence", "Tags", "nodeType",
        key="TIUrl", display="TIUrl"
    )

    # ===================== EDGES (12 types) =====================

    .add_edge("Sent")
    .from_dataframe(sent_edge_df)
    .source(id_column="SenderEmail", node_type="Sender")
    .target(id_column="EmailId", node_type="Email")
    .with_columns(
        "edgeType", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("ReceivedEmail")
    .from_dataframe(received_email_edge_df)
    .source(id_column="EmailId", node_type="Email")
    .target(id_column="UserEmail", node_type="User")
    .with_columns(
        "edgeType", "DeliveryLocation", "DeliveryAction", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("ContainsUrl")
    .from_dataframe(contains_url_edge_df)
    .source(id_column="EmailId", node_type="Email")
    .target(id_column="UrlId", node_type="Url")
    .with_columns(
        "edgeType", "UrlLocation", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("HasDomain")
    .from_dataframe(has_domain_edge_df)
    .source(id_column="EmailId", node_type="Email")
    .target(id_column="DomainId", node_type="Domain")
    .with_columns(
        "edgeType", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("HasAttachment")
    .from_dataframe(has_attachment_edge_df)
    .source(id_column="EmailId", node_type="Email")
    .target(id_column="AttachmentId", node_type="Attachment")
    .with_columns(
        "edgeType", "FileName", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("ClickedUrl")
    .from_dataframe(clicked_url_edge_df)
    .source(id_column="UserEmail", node_type="User")
    .target(id_column="UrlId", node_type="Url")
    .with_columns(
        "edgeType", "IsClickedThrough", "ActionType", "IPAddress", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("StoredOn")
    .from_dataframe(stored_on_edge_df)
    .source(id_column="AttachmentId", node_type="Attachment")
    .target(id_column="DeviceId", node_type="Device")
    .with_columns(
        "edgeType", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("DownloadedFile")
    .from_dataframe(downloaded_file_edge_df)
    .source(id_column="UserEmail", node_type="User")
    .target(id_column="AttachmentId", node_type="Attachment")
    .with_columns(
        "edgeType", "FileName", "DeviceName", "FileOriginUrl", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("TriggeredProcess")
    .from_dataframe(triggered_process_edge_df)
    .source(id_column="AttachmentId", node_type="Attachment")
    .target(id_column="ProcessId", node_type="Process")
    .with_columns(
        "edgeType", "FileName", "ProcessCommandLine", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("OnDevice")
    .from_dataframe(on_device_edge_df)
    .source(id_column="ProcessId", node_type="Process")
    .target(id_column="DeviceId", node_type="Device")
    .with_columns(
        "edgeType", "TimeGenerated", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("DomainRelatedTo")
    .from_dataframe(domain_related_edge_df)
    .source(id_column="TIDomain", node_type="ThreatIndicator_Domain")
    .target(id_column="DomainId", node_type="Domain")
    .with_columns(
        "edgeType", "Confidence", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

    .add_edge("UrlRelatedTo")
    .from_dataframe(url_related_edge_df)
    .source(id_column="TIUrl", node_type="ThreatIndicator_Url")
    .target(id_column="UrlId", node_type="Url")
    .with_columns(
        "edgeType", "Confidence", "EdgeKey",
        key="EdgeKey", display="edgeType"
    )

).done()

In [ ]:
# Check the schema of the graph spec to ensure it is correct
phishing_killchain_graph.show_schema()

## Build & Publish

Build the graph from the spec. This validates the schema, prepares the data, and publishes the graph for querying.

In [ ]:
# Build the graph from the spec - this will validate the spec and prepare it for querying
# Alternative: use Graph.prepare() to prepare the graph nodes and edges in the lake
# and then use Graph.publish() to create the graph. You would typically call prepare()
# and publish() separately to understand the cost of Graph API calls triggered by Graph.publish()
# see https://learn.microsoft.com/azure/sentinel/billing?tabs=simplified%2Ccommitment-tiers
phishing_graph = Graph.build(phishing_killchain_graph)